## SCD OCTA Data Audit

We use **we** language because this is for our class.

### Goal
- We check what OCTA files we have per subject ID.
- We generate a CSV index and a JSON summary.
- We compute a simple biomarker proxy (vessel density) from binarized masks.

**Data source:** TIFFs in **`OCTA AI/`** next to **`scd-octa-screening/`** (flat folder — all `.tif` in one directory). The notebook uses the same flat indexing logic as in `detection.ipynb`, because `make index` normally expects one subfolder per subject under `data/raw/scd-data`.

### Our terminology (complete terms, simple meaning)
- **OCTA**: Optical Coherence Tomography Angiography (shows retinal blood vessels / blood flow patterns).
- **OD**: right eye (oculus dexter).
- **OS**: left eye (oculus sinister).
- **SVP**: Superficial Vascular Plexus (more superficial/top vessel layer).
- **DCP**: Deep Capillary Plexus (deeper vessel layer).
- **binarized mask**: black/white vessel map used for pixel-count measurements.

In [1]:
from pathlib import Path

import pandas as pd


def find_scd_octa_repo() -> Path:
    """Walk up from cwd until we find `src/scd_octa` (works when cwd is notebooks/)."""
    p = Path.cwd().resolve()
    for ancestor in [p, *p.parents]:
        if (ancestor / "src" / "scd_octa").is_dir():
            return ancestor
    raise RuntimeError(
        "Could not find scd-octa-screening (no src/scd_octa). "
        "Start Jupyter from inside scd-octa-screening or a parent folder."
    )


REPO_ROOT = find_scd_octa_repo()

# TIFFs: flat `OCTA AI/` next to the repo under Hospital when present
_hospital = REPO_ROOT.parent
if (_hospital / "OCTA AI").is_dir():
    DATA_ROOT = _hospital / "OCTA AI"
else:
    DATA_ROOT = REPO_ROOT / "data" / "raw" / "scd-data"

OUT_DIR = REPO_ROOT / "results"

DATA_ROOT.resolve(), OUT_DIR.resolve()

(PosixPath('/Users/tripa/Desktop/Projects/Hospital/OCTA AI'),
 PosixPath('/Users/tripa/Desktop/Projects/Hospital/scd-octa-screening/results'))

In [2]:
# Package imports — add src/ using REPO_ROOT (same discovery as cell 1 if not run yet)
import sys
from pathlib import Path

if "REPO_ROOT" not in globals():
    _cwd = Path.cwd().resolve()
    REPO_ROOT = next(
        (a for a in [_cwd, *_cwd.parents] if (a / "src" / "scd_octa").is_dir()),
        None,
    )
    if REPO_ROOT is None:
        raise RuntimeError("Run the previous cell first, or cd into scd-octa-screening.")

_src = (REPO_ROOT / "src").resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from dataclasses import asdict

from scd_octa.compute_biomarkers import read_tif_gray, vessel_density_from_mask
from scd_octa.io import parse_octa_filename, records_to_dataframe
from scd_octa.types import SubjectRecord

In [3]:
# Build index from flat OCTA AI folder (same aggregation as nested scd-data, filename-driven)
import json


def _count_nonempty(row: dict[str, str], cols: list[str]) -> int:
    return sum(1 for c in cols if row.get(c, "").strip() != "")


def iter_octa_files_flat(data_root: Path):
    paths = []
    for ext in (".tif", ".tiff"):
        paths.extend(data_root.glob(f"*{ext}"))
    for p in sorted(paths, key=lambda x: x.name.lower()):
        parsed = parse_octa_filename(p)
        if parsed is not None:
            yield parsed


def build_subject_records_flat(data_root: Path) -> list[SubjectRecord]:
    records: dict[str, SubjectRecord] = {}

    def get_record(sid: str) -> SubjectRecord:
        if sid not in records:
            records[sid] = SubjectRecord(subject_id=sid)
        return records[sid]

    for f in iter_octa_files_flat(data_root):
        r = get_record(f.subject_id)
        key = (f.eye, f.plexus, f.kind)
        if key == ("OD", "SVP", "original"):
            records[f.subject_id] = SubjectRecord(**{**asdict(r), "od_svp": f.path})
        elif key == ("OD", "DCP", "original"):
            records[f.subject_id] = SubjectRecord(**{**asdict(r), "od_dcp": f.path})
        elif key == ("OS", "SVP", "original"):
            records[f.subject_id] = SubjectRecord(**{**asdict(r), "os_svp": f.path})
        elif key == ("OS", "DCP", "original"):
            records[f.subject_id] = SubjectRecord(**{**asdict(r), "os_dcp": f.path})
        elif key == ("OD", "SVP", "binarized"):
            records[f.subject_id] = SubjectRecord(**{**asdict(r), "od_svp_bin": f.path})
        elif key == ("OD", "DCP", "binarized"):
            records[f.subject_id] = SubjectRecord(**{**asdict(r), "od_dcp_bin": f.path})
        elif key == ("OS", "SVP", "binarized"):
            records[f.subject_id] = SubjectRecord(**{**asdict(r), "os_svp_bin": f.path})
        elif key == ("OS", "DCP", "binarized"):
            records[f.subject_id] = SubjectRecord(**{**asdict(r), "os_dcp_bin": f.path})

    return [records[k] for k in sorted(records.keys(), key=lambda x: int(x))]


OUT_DIR.mkdir(parents=True, exist_ok=True)

records = build_subject_records_flat(DATA_ROOT)
idx_df = records_to_dataframe(records)

original_cols = ["od_svp", "od_dcp", "os_svp", "os_dcp"]
bin_cols = ["od_svp_binarized", "od_dcp_binarized", "os_svp_binarized", "os_dcp_binarized"]
idx_df["n_original"] = idx_df.apply(lambda r: _count_nonempty(r.to_dict(), original_cols), axis=1)
idx_df["n_binarized"] = idx_df.apply(lambda r: _count_nonempty(r.to_dict(), bin_cols), axis=1)
idx_df["n_total"] = idx_df["n_original"] + idx_df["n_binarized"]
idx_df["is_complete_8"] = idx_df["n_total"] == 8

csv_path = OUT_DIR / "dataset_index.csv"
idx_df.to_csv(csv_path, index=False)

summary = {
    "data_root": str(DATA_ROOT.resolve()),
    "n_subjects_with_any_tif": int(idx_df.shape[0]),
    "n_complete_subjects_8": int(idx_df["is_complete_8"].sum()),
    "subjects_complete": idx_df.loc[idx_df["is_complete_8"], "subject_id"].tolist(),
    "subjects_incomplete": idx_df.loc[~idx_df["is_complete_8"], "subject_id"].tolist(),
    "note": "Flat OCTA AI folder: subjects aggregated by filename pattern (see scd_octa.io.parse_octa_filename).",
}
(OUT_DIR / "dataset_summary.json").write_text(json.dumps(summary, indent=2) + "\n")

print(f"Wrote: {csv_path}")
print(f"Wrote: {OUT_DIR / 'dataset_summary.json'}")

Wrote: /Users/tripa/Desktop/Projects/Hospital/scd-octa-screening/results/dataset_index.csv
Wrote: /Users/tripa/Desktop/Projects/Hospital/scd-octa-screening/results/dataset_summary.json


In [4]:
# Load and show the index
idx = pd.read_csv(OUT_DIR / 'dataset_index.csv')
idx

,subject_id,od_svp,od_dcp,os_svp,os_dcp,od_svp_binarized,od_dcp_binarized,os_svp_binarized,os_dcp_binarized,n_original,n_binarized,n_total,is_complete_8
0,4,NaN,NaN,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,NaN,NaN,NaN,NaN,2,0,2,False
1,5,NaN,NaN,NaN,NaN,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0,4,4,False
2,7,NaN,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,NaN,NaN,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,NaN,NaN,NaN,1,1,2,False
3,11,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,4,4,8,True
4,12,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,4,4,8,True
5,13,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,4,4,8,True
6,15,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,4,4,8,True
7,17,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,4,4,8,True
8,19,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,4,4,8,True
9,26,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,4,4,8,True


In [5]:
# Compute mask-based vessel density features (flat DATA_ROOT — same logic as compute_biomarkers CLI)
import numpy as np

records_bio = build_subject_records_flat(DATA_ROOT)
rows: list[dict] = []
for r in records_bio:
    row: dict = {"subject_id": r.subject_id}
    for name, p in [
        ("od_svp", r.od_svp_bin),
        ("od_dcp", r.od_dcp_bin),
        ("os_svp", r.os_svp_bin),
        ("os_dcp", r.os_dcp_bin),
    ]:
        if p is None:
            row[f"{name}_vessel_density"] = np.nan
            row[f"{name}_mask_path"] = ""
            continue
        try:
            mask = read_tif_gray(p)
            row[f"{name}_vessel_density"] = vessel_density_from_mask(mask)
            row[f"{name}_mask_path"] = str(p)
            row[f"{name}_mask_h"] = int(mask.shape[0])
            row[f"{name}_mask_w"] = int(mask.shape[1])
        except Exception as e:
            row[f"{name}_vessel_density"] = np.nan
            row[f"{name}_mask_path"] = str(p)
            row[f"{name}_error"] = f"{type(e).__name__}: {e}"
    rows.append(row)

bio_df = pd.DataFrame(rows)
bio_path = OUT_DIR / "biomarkers_vessel_density.csv"
bio_df.to_csv(bio_path, index=False)
print(f"Wrote: {bio_path}")

Wrote: /Users/tripa/Desktop/Projects/Hospital/scd-octa-screening/results/biomarkers_vessel_density.csv


In [6]:
bio = pd.read_csv(OUT_DIR / 'biomarkers_vessel_density.csv')
bio

,subject_id,od_svp_vessel_density,od_svp_mask_path,od_dcp_vessel_density,od_dcp_mask_path,os_svp_vessel_density,os_svp_mask_path,os_dcp_vessel_density,os_dcp_mask_path,od_svp_mask_h,od_svp_mask_w,od_dcp_mask_h,od_dcp_mask_w,os_svp_mask_h,os_svp_mask_w,os_dcp_mask_h,os_dcp_mask_w
0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5,0.406667,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.311674,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.407647,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.301276,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,887.0,887.0,886.0,888.0,884.0,884.0,888.0,884.0
2,7,0.379471,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,NaN,NaN,NaN,NaN,NaN,NaN,884.0,888.0,NaN,NaN,NaN,NaN,NaN,NaN
3,11,0.224257,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.222071,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.287754,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.265653,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,891.0,888.0,884.0,888.0,889.0,888.0,884.0,891.0
4,12,0.270343,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.200247,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.342274,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.264195,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,884.0,889.0,885.0,889.0,886.0,891.0,887.0,888.0
5,13,0.441149,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.331705,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.374048,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.263802,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,887.0,887.0,887.0,889.0,887.0,888.0,884.0,891.0
6,15,0.346564,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.242663,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.401659,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.300667,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,889.0,891.0,889.0,892.0,888.0,890.0,889.0,891.0
7,17,0.400468,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.322534,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.358779,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.269873,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,888.0,890.0,888.0,890.0,885.0,890.0,889.0,888.0
8,19,0.269451,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.209707,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.208966,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.166452,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,885.0,886.0,883.0,888.0,886.0,889.0,891.0,888.0
9,26,0.274489,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.306327,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.258796,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,0.266798,/Users/tripa/Desktop/Projects/Hospital/OCTA AI...,885.0,941.0,887.0,945.0,889.0,890.0,885.0,889.0


In [7]:
# Build / refresh `results/clinical_labels.csv`.
from pathlib import Path

# **Filled from code (actual):** counts and flags merged from `dataset_index.csv` after indexing OCTA AI.
# **Left for you to type:** genotype, scm_present, scr_stage, visit_date, clinical_notes, data_availability_notes.
# Re-runs refresh **computed** columns only; any non-empty manual fields you already saved are preserved.

COLS_COMPUTED = [
    "n_octa_views",
    "n_original_views",
    "n_binarized_masks",
    "full_8_view_set",
]
COLS_CHART = ["genotype", "scm_present", "scr_stage", "visit_date", "clinical_notes"]
COLS_MANUAL = [*COLS_CHART, "data_availability_notes"]
COLS = ["subject_id", *COLS_COMPUTED, *COLS_MANUAL]


def _sort_subject_ids(ids: list) -> list[str]:
    def _key(s: str):
        s = str(s)
        return (0, int(s)) if s.isdigit() else (1, s)

    return sorted(map(str, ids), key=_key)


def sync_clinical_labels_csv() -> Path:
    idx_path = OUT_DIR / "dataset_index.csv"
    idx = pd.read_csv(idx_path)
    idx["subject_id"] = idx["subject_id"].astype(str)
    want_ids = _sort_subject_ids(idx["subject_id"].unique())

    out_path = OUT_DIR / "clinical_labels.csv"
    existing_manual: dict[str, dict[str, str]] = {}
    if out_path.is_file():
        prev = pd.read_csv(out_path)
        for _, row in prev.iterrows():
            sid = str(row["subject_id"])
            per: dict[str, str] = {}
            for c in COLS_MANUAL:
                if c not in prev.columns:
                    continue
                v = row.get(c, "")
                if pd.isna(v):
                    continue
                v = str(v).strip()
                if v:
                    per[c] = v
            if per:
                existing_manual[sid] = per

    rows = []
    for sid in want_ids:
        r = idx[idx["subject_id"] == sid].iloc[0]
        n_tot = int(r["n_total"])
        n_orig = int(r["n_original"])
        n_bin = int(r["n_binarized"])
        complete = bool(r["is_complete_8"])

        manual = {c: "" for c in COLS_MANUAL}
        manual.update(existing_manual.get(sid, {}))

        rows.append(
            {
                "subject_id": sid,
                "n_octa_views": n_tot,
                "n_original_views": n_orig,
                "n_binarized_masks": n_bin,
                "full_8_view_set": complete,
                **manual,
            }
        )

    pd.DataFrame(rows, columns=COLS).to_csv(out_path, index=False)
    print(
        f"Wrote {out_path} ({len(rows)} subjects). "
        "Computed counts refreshed from dataset_index; manual columns preserved when non-empty."
    )
    return out_path


sync_clinical_labels_csv()

Wrote /Users/tripa/Desktop/Projects/Hospital/scd-octa-screening/results/clinical_labels.csv (19 subjects). Computed counts refreshed from dataset_index; manual columns preserved when non-empty.


PosixPath('/Users/tripa/Desktop/Projects/Hospital/scd-octa-screening/results/clinical_labels.csv')

### What we are missing (and why we need it)

Even if our image files are perfect, we cannot train a real disease detector without labels.

We keep **`results/clinical_labels.csv`** in sync via the previous cell.

**Computed from indexing (filled automatically — no fabrication):**

| Column | Meaning |
|--------|---------|
| `subject_id` | Matches `dataset_index.csv` |
| `n_octa_views` | Count of populated view slots (`n_total`, max 8) |
| `n_original_views` | Count of raw en-face TIFFs indexed |
| `n_binarized_masks` | Count of binarized mask TIFFs indexed |
| `full_8_view_set` | Whether all eight expected files were found |

**You fill in (not filled by code):**

| Column | Meaning |
|--------|---------|
| `genotype` | e.g. HbSS, HbSC |
| `scm_present` | `0` absent / `1` present where documented |
| `scr_stage` | Your SCR grading |
| `visit_date` | `YYYY-MM-DD` if tracking visits |
| `clinical_notes` | Other abstraction-approved notes |
| `data_availability_notes` | Optional free text (e.g. why views are missing); left blank by the sync cell |

Indexing stats are already in `n_octa_views` / `n_*_views` / `full_8_view_set` — no duplicate prose is written into the CSV.

Re-running the sync cell refreshes computed columns only; non-empty manual fields are preserved per `subject_id`.

When screening labels are finalized, add **`label`** / **`octa_abnormal_label`** into **`dataset_index.csv`** as in the README, or join this CSV in pandas before training.

eye-level outcomes can be added later as extra rows or columns (`eye` = OD/OS) if your charts are per-eye.